## Inference with Flute Music

Now that we have an ornament classifier trained, let us see how well it performs on unseen hindustani flute music

In [ ]:
import numpy as np
from pathlib import Path

from hcm_transcription.segmentation import StaticWindower, segment_phrases

from model_definitions import OrnamentTCN

In [12]:
def _load_audio_mono_16k(path: Path) -> tuple[np.ndarray, int]:
    try:
        import librosa
    except Exception as e:  # pragma: no cover
        raise RuntimeError(
            "librosa is required to load audio. Install requirements.txt first."
        ) from e

    y, sr = librosa.load(str(path), sr=16000, mono=True)
    y = np.asarray(y, dtype=np.float32)
    return y, int(sr)


def _predict_f0_swiftf0(
    audio_path: Path,
    *,
    fmin: float = 46.875,
    fmax: float = 2093.75,
    confidence_threshold: float = 0.9,
) -> tuple[np.ndarray, np.ndarray]:
    try:
        from swift_f0 import SwiftF0  # type: ignore
    except Exception as e:  # pragma: no cover
        raise RuntimeError(
            "swift-f0 is required but not installed. Try: pip install swift-f0"
        ) from e

    detector = SwiftF0(
        fmin=float(fmin),
        fmax=float(fmax),
        confidence_threshold=float(confidence_threshold),
    )
    result = detector.detect_from_file(str(audio_path))
    times_s = np.asarray(result.timestamps, dtype=np.float64)
    pitch_hz = np.asarray(result.pitch_hz, dtype=np.float64)
    voicing = np.asarray(result.voicing, dtype=bool)
    f0_hz = np.where(voicing, pitch_hz, 0.0)
    return times_s, f0_hz


def _build_static_windower(sample_period_s: float) -> StaticWindower:
    frame_samples = max(8, int(round(0.250 / sample_period_s)))
    hop_samples = max(1, int(round(0.022 / sample_period_s)))
    return StaticWindower(frame_samples=frame_samples, hop_samples=hop_samples, ma_window=5)


def _find_unstable_regions(
    instability: np.ndarray,
    *,
    instability_threshold: float,
    min_merge_gap: int = 3,
) -> list[tuple[int, int]]:
    if len(instability) == 0:
        return []

    above = instability > float(instability_threshold)
    above = np.where(np.isnan(instability), False, above)

    regions: list[tuple[int, int]] = []
    in_region = False
    start = 0
    for i in range(len(above)):
        if above[i] and not in_region:
            start = i
            in_region = True
        elif not above[i] and in_region:
            regions.append((start, i))
            in_region = False
    if in_region:
        regions.append((start, len(above)))

    merged: list[tuple[int, int]] = []
    for region in regions:
        if merged and (region[0] - merged[-1][1]) <= int(min_merge_gap):
            merged[-1] = (merged[-1][0], region[1])
        else:
            merged.append(region)

    return merged

In [13]:
audio_path = "flute-recordings/Bandish_Teentaal_1.mp3"
f0_times, f0_hz = _predict_f0_swiftf0(audio_path)
sample_period_s = float(np.median(np.diff(f0_times)))
quantized_hz = f0_hz.copy()
swara_labels = np.where(quantized_hz > 0, "P", "REST").astype(str)

In [14]:
windower = _build_static_windower(sample_period_s)
phrases = segment_phrases(
    f0_hz=f0_hz,
    times=f0_times,
    quantized_hz=quantized_hz,
    swara_labels=swara_labels,
    sample_period_s=sample_period_s,
)

print(
    f"sample period={sample_period_s:.4f}s | frame={windower.frame_samples} samples "
    f"({windower.frame_samples * sample_period_s:.3f}s) | hop={windower.hop_samples} samples "
    f"({windower.hop_samples * sample_period_s:.3f}s)"
 )

sample period=0.0160s | frame=16 samples (0.256s) | hop=1 samples (0.016s)


In [15]:
total_regions = 0
tcn_segments: list[dict] = []
phrase_debug: list[dict] = []

for i, phrase in enumerate(phrases):
    phrase_times_full = f0_times[phrase.start_idx : phrase.end_idx]
    phrase_quant_full = phrase.quantized_hz
    voiced_mask = phrase_quant_full != 0
    phrase_times = phrase_times_full[voiced_mask]
    phrase_quant = phrase_quant_full[voiced_mask]
    phrase_f0 = phrase.f0_hz[voiced_mask]

    instab = windower.compute_instability(phrase_quant)
    if len(instab) == 0:
        phrase_debug.append({
            "phrase_index": i,
            "voiced_samples": int(len(phrase_quant)),
            "instability_frames": 0,
            "threshold": None,
            "regions": 0,
        })
        continue

    phrase_threshold = float(np.nanmean(instab) + np.nanstd(instab))
    regions = _find_unstable_regions(
        instab,
        instability_threshold=phrase_threshold,
        min_merge_gap=3,
    )
    total_regions += len(regions)

    phrase_debug.append({
        "phrase_index": i,
        "voiced_samples": int(len(phrase_quant)),
        "instability_frames": int(len(instab)),
        "max_instability": float(np.nanmax(instab)),
        "threshold": phrase_threshold,
        "regions": int(len(regions)),
    })

    hop = int(windower.hop_samples)
    for start_frame, end_frame in regions:
        seg_start = int(start_frame * hop)
        seg_end = int(min(end_frame * hop, len(phrase_f0)))
        if seg_end <= seg_start:
            continue

        seg_f0 = phrase_f0[seg_start:seg_end]
        if len(seg_f0) == 0:
            continue

        pitch_curve = np.log2(np.asarray(seg_f0, dtype=np.float32))
        time_s = float(phrase_times[seg_start]) if seg_start < len(phrase_times) else float(phrase.start_time)
        time_e = float(phrase_times[min(seg_end - 1, len(phrase_times) - 1)]) if len(phrase_times) else float(phrase.end_time)
        tcn_segments.append(
            {
                "label": None,
                "time_s": time_s,
                "time_e": time_e,
                "duration": float(time_e - time_s),
                "pitch_curve": pitch_curve,
                "phrase_index": i,
                "phrase_event_start_idx": int(seg_start),
                "phrase_event_end_idx": int(seg_end),
                "instability_threshold": phrase_threshold,
            }
        )

print(f"total_regions={total_regions}")

total_regions=33


In [16]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

def prepare_tcn_input(segments, fixed_length=117, min_length=20, verbose=True):
    """
    Prepare fixed-length arrays for TCN inference or training.


    Unlabeled segments are kept with y=-1 so inference can still run.
    """
    label_map = {"Kan": 0, "Meend": 1, "Murki": 2, "Andolan": 3}
    X, y = [], []
    skipped = 0

    segments = [s for s in segments if len(s.get("pitch_curve", [])) >= int(min_length)]

    for seg in segments:
        curve = np.asarray(seg["pitch_curve"], dtype=np.float32)
        if curve.ndim != 1:
            skipped += 1
            continue

        label = seg.get("label")
        if label is None:
            y.append(-1)
        else:
            if label not in label_map:
                skipped += 1
                continue
            y.append(int(label_map[label]))

        std = float(curve.std())
        if std > 0:
            curve = (curve - float(curve.mean())) / std

        if len(curve) < int(fixed_length):
            curve = np.pad(curve, (0, int(fixed_length) - len(curve)), mode="constant")
        else:
            curve = curve[: int(fixed_length)]

        X.append(curve)

    if not X:
        X_arr = np.zeros((0, 1, int(fixed_length)), dtype=np.float32)
        y_arr = np.zeros((0,), dtype=np.int64)
        if verbose:
            print(f"X shape: {X_arr.shape}")
            print(f"y shape: {y_arr.shape}")
            print(f"Skipped: {skipped}")
            print(f"Label distribution: {{ {', '.join([f'{k}: 0' for k in label_map])} }}")
        return X_arr, y_arr, label_map

    X_arr = np.stack(X, axis=0)[:, np.newaxis, :]
    y_arr = np.asarray(y, dtype=np.int64)

    if verbose:
        print(f"X shape: {X_arr.shape}")
        print(f"y shape: {y_arr.shape}")
        print(f"Skipped: {skipped}")
        print(f"Label distribution: { {k: int((y_arr == v).sum()) for k, v in label_map.items()} }")

    return X_arr, y_arr, label_map


X, y, label_map = prepare_tcn_input(
    tcn_segments,
    fixed_length=117,
    min_length=20,
    verbose=True,
)

X shape: (5, 1, 117)
y shape: (5,)
Skipped: 0
Label distribution: {'Kan': 0, 'Meend': 0, 'Murki': 0, 'Andolan': 0}


In [17]:
model = OrnamentTCN()

In [18]:
checkpoint = torch.load('checkpoints/ornament_tcn.pth')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

OrnamentTCN(
  (input_proj): Conv1d(1, 32, kernel_size=(1,), stride=(1,))
  (tcn): Sequential(
    (0): ResidualBlock(
      (conv1): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(2,))
      (conv2): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(2,))
      (relu): ReLU()
      (dropout): Dropout(p=0.2, inplace=False)
      (norm1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (norm2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): ResidualBlock(
      (conv1): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(2,))
      (conv2): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(2,))
      (relu): ReLU()
      (dropout): Dropout(p=0.2, inplace=False)
      (norm1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (norm2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )

In [19]:
if X.shape[0] == 0:
    raise ValueError("No TCN segments available for inference. Re-run Cells 4-7 and inspect phrase_debug.")

idx_to_label = {v: k for k, v in label_map.items()}
infer_ds = TensorDataset(torch.from_numpy(X).to(dtype=torch.float32))
infer_loader = DataLoader(infer_ds, batch_size=32, shuffle=False)

all_probs = []
all_pred_idx = []

with torch.no_grad():
    for (xb,) in infer_loader:
        logits = model(xb)
        probs = torch.softmax(logits, dim=1)
        all_probs.append(probs.cpu())
        all_pred_idx.append(probs.argmax(dim=1).cpu())

probs = torch.cat(all_probs, dim=0).numpy()
pred_idx = torch.cat(all_pred_idx, dim=0).numpy()
pred_labels = [idx_to_label[int(i)] for i in pred_idx]

predicted_segments = []
for seg, pred_label, prob_row in zip(tcn_segments, pred_labels, probs):
    predicted_segments.append(
        {
            **seg,
            "predicted_label": pred_label,
            "predicted_probs": prob_row.tolist(),
            "predicted_confidence": float(np.max(prob_row)),
        }
    )

predicted_segments[:3]

[{'label': None,
  'time_s': 3.36796875,
  'time_e': 3.36796875,
  'duration': 0.0,
  'pitch_curve': array([6.62785], dtype=float32),
  'phrase_index': 0,
  'phrase_event_start_idx': 86,
  'phrase_event_end_idx': 87,
  'instability_threshold': 0.011346626551846886,
  'predicted_label': 'Kan',
  'predicted_probs': [0.8266639709472656,
   0.0335557758808136,
   0.07000166922807693,
   0.06977856904268265],
  'predicted_confidence': 0.8266639709472656},
 {'label': None,
  'time_s': 4.23196875,
  'time_e': 4.50396875,
  'duration': 0.27200000000000024,
  'pitch_curve': array([8.69069 , 8.69069 , 8.69069 , 8.69069 , 8.690689, 8.690689,
         8.690688, 8.690689, 8.690689, 8.69069 , 8.69069 ], dtype=float32),
  'phrase_index': 0,
  'phrase_event_start_idx': 123,
  'phrase_event_end_idx': 134,
  'instability_threshold': 0.011346626551846886,
  'predicted_label': 'Kan',
  'predicted_probs': [0.8876132965087891,
   0.04684697464108467,
   0.06509463489055634,
   0.0004450834821909666],
  'pre

In [20]:
from IPython.display import Audio, display

audio_waveform, audio_sr = _load_audio_mono_16k(Path(audio_path))

def play_predicted_segments(predicted_segments, max_examples=10, pad_s=0.15):
    shown = min(max_examples, len(predicted_segments))
    print(f"Showing {shown} predicted segments")

    for i, seg in enumerate(predicted_segments[:shown]):
        start_s = max(0.0, float(seg["time_s"]) - float(pad_s))
        end_s = min(len(audio_waveform) / audio_sr, float(seg["time_e"]) + float(pad_s))
        start_idx = int(start_s * audio_sr)
        end_idx = int(end_s * audio_sr)

        print(
            f"{i + 1}. {seg['predicted_label']} | confidence={seg['predicted_confidence']:.3f} | "
            f"segment={seg['time_s']:.2f}s-{seg['time_e']:.2f}s | "
            f"playback={start_s:.2f}s-{end_s:.2f}s"
        )
        display(Audio(audio_waveform[start_idx:end_idx], rate=audio_sr))

play_predicted_segments(predicted_segments, max_examples=10, pad_s=0.15)

Showing 5 predicted segments
1. Kan | confidence=0.827 | segment=3.37s-3.37s | playback=3.22s-3.52s


2. Kan | confidence=0.888 | segment=4.23s-4.50s | playback=4.08s-4.65s


3. Kan | confidence=0.877 | segment=4.94s-5.11s | playback=4.79s-5.26s


4. Kan | confidence=0.564 | segment=5.62s-5.77s | playback=5.47s-5.92s


5. Kan | confidence=0.686 | segment=12.65s-12.66s | playback=12.50s-12.81s
